In [ ]:
import pandas as pd 
import pickle as pl
import numpy as np
import plotly.express as px
import numpy as np
from EDA_1 import procesado_datos
from calcular_distancia import localizacion_centro_provincia, calcular_distancia_tiempo


In this notebook, we present a preliminary test of the model using data from May. We are aware of potential data leakage due to overlapping IDs and timestamps across months. Since the model is static and the objective here is exploratory, we use this setup as an approximation rather than for formal evaluation. To reduce bias introduced by repeated entries, we concatenate data from all years and apply a unique() operation by ID, allowing the model to sample prices in a less redundant and more randomized way. However, we acknowledge that a proper evaluation would require stricter controls to avoid leakage and accurately measure generalization performance.

In [3]:
df0 = pd.read_excel('/Users/silvanaruizmedina/Desktop/TFM/Eventos/DATOS/datos0524/Consulta_suicidios_presentes_090524.xlsx')

In [51]:
df0p = procesado_datos(df0)

Coordenadas inválidas para centro 'Murcia II' y provincia 'nan': [37.9912566, -1.0602803, None, None]
Error en el intento 1: float() argument must be a string or a real number, not 'NoneType'
Error en el intento 2: float() argument must be a string or a real number, not 'NoneType'
Error en el intento 3: float() argument must be a string or a real number, not 'NoneType'
Coordenadas inválidas para centro 'Teixeiro (A Coruña)' y provincia 'nan': [43.1413871, -8.0376749, None, None]
Error en el intento 1: float() argument must be a string or a real number, not 'NoneType'
Error en el intento 2: float() argument must be a string or a real number, not 'NoneType'
Error en el intento 3: float() argument must be a string or a real number, not 'NoneType'
Diccionario de distancias y tiempos actualizado y guardado.


In [52]:
df0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46291 entries, 0 to 46290
Data columns (total 89 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ID_INTERNO                           46291 non-null  int64  
 1   CODIGO_CENTRO_INGRESO                46291 non-null  int64  
 2   NOMBRE_CENTRO                        46291 non-null  object 
 3   PROVINCIA_CENTRO                     46291 non-null  object 
 4   MUNICIPIO_CENTRO                     46291 non-null  object 
 5   PROVINCIA_RESIDENCIA                 42304 non-null  object 
 6   NACIONALIDAD                         46291 non-null  object 
 7   FECHA_NACIMIENTO                     46291 non-null  object 
 8   SEXO                                 46291 non-null  object 
 9   NUM_ALTAS_LIBERTAD                   46291 non-null  int64  
 10  FECHA_ULTIMO_INGRESO                 46291 non-null  object 
 11  FECHA_INGRESO_CENTRO_ACTUAL 

In [ ]:
# Cargamos el modelo
with open('modelo_carceles.pkl', 'rb') as f:
    obj = pl.load(f)
    meta_model = obj['modelo']
    scaler = obj['scaler']
    features= obj['features']
    rf= obj['rf']
    gb= obj['gb']
    hist= obj['hist']

In [ ]:
X_input = df0p[features]

# Predicciones de los modelos base
pred_rf = rf.predict_proba(X_input)[:, 1]
pred_gb = gb.predict_proba(X_input)[:, 1]
pred_hist = hist.predict_proba(X_input)[:, 1]

# Construir matriz de entrada para el meta-modelo
X_meta = np.column_stack((pred_rf, pred_gb, pred_hist))

# Escalar datos
X_meta_scaled = scaler.transform(X_meta)

# Predicción final
pred_clase = meta_model.predict(X_meta_scaled)
pred_proba = meta_model.predict_proba(X_meta_scaled)[:, 1]

In [ ]:
# Creamos dataframe para hacer la comparción
df = pd.DataFrame()
df['ID']= df0p['ID_INTERNO']
df['Clase']= df0p['CLASE']
df['Clase_predicha']= pred_clase
df['Centro'] = df0['NOMBRE_CENTRO']


In [ ]:
ids_error = df.loc[df['Clase'] != df['Clase_predicha'], 'ID']
df_errores = df[df['Clase'] != df['Clase_predicha']]
errores_falsos_positivos = df_errores[(df_errores['Clase'] == 0) & (df_errores['Clase_predicha'] == 1)]
print("Total de errores:", len(df_errores))
print("Errores donde se predijo 1 pero era 0 (falsos positivos):", len(errores_falsos_positivos))
falsos_negativos = df[(df['Clase'] == 1) & (df['Clase_predicha'] == 0)]
print("Total de falsos negativos (Clase real = 1, predicha = 0):", len(falsos_negativos), '/',len(df[df['Clase']==1]))
print("IDs de falsos negativos:")

Total de errores: 4314
Errores donde se predijo 1 pero era 0 (falsos positivos): 4164
Total de falsos negativos (Clase real = 1, predicha = 0): 150 / 658
IDs de falsos negativos:


#### The section below generates a figure showing the number of false positives and a figure that shows the proportion of false positives relative to the total population of that center.

In [ ]:
# Obtener los datos y ordenarlos de mayor a menor
fp_por_centro = errores_falsos_positivos['Centro'].value_counts().sort_values(ascending=False)

# Crear DataFrame para Plotly
df_plot = fp_por_centro.reset_index()
df_plot.columns = ['Centro', 'Falsos_Positivos']

# Número de divisiones
n_divisiones = 8
centros_totales = len(df_plot)
tamaño_bloque = int(np.ceil(centros_totales / n_divisiones))

# Crear 8 gráficos ordenados visualmente
for i in range(n_divisiones):
    start_idx = i * tamaño_bloque
    end_idx = min((i + 1) * tamaño_bloque, centros_totales)
    subset = df_plot.iloc[start_idx:end_idx].copy()

    # Ordenar de mayor a menor y fijar el orden del eje Y
    subset = subset.sort_values(by='Falsos_Positivos', ascending=False)
    subset['Centro'] = pd.Categorical(subset['Centro'], categories=subset['Centro'], ordered=True)

    fig = px.bar(
        subset,
        x='Falsos_Positivos',
        y='Centro',
        orientation='h',
        title=f'Falsos positivos por centro (Bloque {i + 1})',
        labels={'Falsos_Positivos': 'Cantidad de falsos positivos', 'Centro': 'Centro penitenciario'},
        color_discrete_sequence=['salmon']
    )

    fig.update_layout(
    yaxis=dict(
        tickfont=dict(size=10),
        categoryorder='total ascending'  # Fuerza el orden de mayor a menor en barras horizontales
    ),
    margin=dict(l=100, r=20, t=40, b=40),
    height=400)


    fig.show()

In [72]:
import plotly.express as px
import pandas as pd
import numpy as np

# Paso 1: Calcular falsos positivos por centro
fp_por_centro = errores_falsos_positivos['Centro'].value_counts().sort_values(ascending=False)
df_fp = fp_por_centro.reset_index()
df_fp.columns = ['Centro', 'Falsos_Positivos']

# Paso 2: Calcular total de internos por centro (usando el DataFrame completo)
total_por_centro = df['Centro'].value_counts().reset_index()
total_por_centro.columns = ['Centro', 'Total_Presos']

# Paso 3: Unir ambos
df_plot = df_fp.merge(total_por_centro, on='Centro', how='left')

# Paso 4: Calcular % de falsos positivos sobre total de presos en cada centro
df_plot['%_Falsos_Positivos'] = (df_plot['Falsos_Positivos'] / df_plot['Total_Presos']) * 100

# Paso 5: Dividir en bloques para graficar
n_divisiones = 8
centros_totales = len(df_plot)
tamaño_bloque = int(np.ceil(centros_totales / n_divisiones))

# Paso 6: Crear gráficos por bloques
for i in range(n_divisiones):
    start_idx = i * tamaño_bloque
    end_idx = min((i + 1) * tamaño_bloque, centros_totales)
    subset = df_plot.iloc[start_idx:end_idx].copy()

    # Ordenar de mayor a menor por % de falsos positivos
    subset = subset.sort_values(by='%_Falsos_Positivos', ascending=False)
    subset['Centro'] = pd.Categorical(subset['Centro'], categories=subset['Centro'], ordered=True)

    fig = px.bar(
        subset,
        x='%_Falsos_Positivos',
        y='Centro',
        orientation='h',
        title=f'Porcentaje de falsos positivos por centro (Bloque {i + 1})',
        labels={'%_Falsos_Positivos': '% de falsos positivos', 'Centro': 'Centro penitenciario'},
        color_discrete_sequence=['salmon'],
        hover_data=['Falsos_Positivos', 'Total_Presos']
    )

    fig.update_layout(
        yaxis=dict(
            tickfont=dict(size=10),
            categoryorder='total ascending'
        ),
        margin=dict(l=100, r=20, t=40, b=40),
        height=400
    )

    fig.show()

In [73]:
# Clasificar los centros según el porcentaje de falsos positivos
condiciones = [
    df_plot['%_Falsos_Positivos'] < 5,
    (df_plot['%_Falsos_Positivos'] >= 5) & (df_plot['%_Falsos_Positivos'] < 10),
    (df_plot['%_Falsos_Positivos'] >= 10) & (df_plot['%_Falsos_Positivos'] < 20)
]

categorias = ['< 5%', '5–10%', '10–20%']

df_plot['Rango_FP'] = np.select(condiciones, categorias, default='≥ 20%')

# Contar cuántos centros hay en cada categoría
tabla_resumen = df_plot['Rango_FP'].value_counts().sort_index()

# Mostrar tabla
print("📊 Número de centros por rango de % de falsos positivos:\n")
print(tabla_resumen)

📊 Número de centros por rango de % de falsos positivos:

Rango_FP
10–20%    23
5–10%     54
< 5%       7
Name: count, dtype: int64
